# Clozapine cardiotoxicity: transcriptomic reanalysis

Reproduction pipeline for the hiPSC-CM and rat validation analyses.

**Accessions:** GSE262419 (discovery, hiPSC-CM TempO-Seq) · GSE244740 (human validation) · GSE59905 (rat heart, DrugMatrix)

**Scope note.** Transcriptomic analyses only. The FAERS disproportionality statistics in
Table 1 came from the OpenVigil 2.1 web interface and cannot be regenerated as code.

**No GEOquery.** GEOquery requires the XML package, which will not build against the
mixed anaconda/system libxml2 on this cluster. Everything it did here is plain HTTP plus
text parsing, handled below in base R with no compiled dependencies.

**How to use this.** Run one cell at a time, top to bottom. Cells marked INSPECT stop and
print structure so you can confirm field names before the next step assumes them. Do not
use Run All. Every target value from the manuscript is asserted at the end of its section,
so a mismatch surfaces immediately.

## 0. Environment

In [1]:
lib <- file.path(Sys.getenv("HOME"), "Rlibs", paste0("R-", getRversion()))
dir.create(lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(lib, .libPaths()))

need <- c("BiocManager","ggplot2","dplyr","tidyr","readr","stringr")
for (p in need) if (!requireNamespace(p, quietly=TRUE))
  install.packages(p, lib=lib, repos="https://cloud.r-project.org", Ncpus=4)

bioc <- c("DESeq2","limma","fgsea","msigdbr")     # GEOquery deliberately omitted
for (p in bioc) if (!requireNamespace(p, quietly=TRUE))
  BiocManager::install(p, lib=lib, update=FALSE, ask=FALSE, Ncpus=4)

suppressPackageStartupMessages({
  library(DESeq2); library(limma); library(msigdbr)
  library(ggplot2); library(dplyr); library(tidyr); library(stringr)
})

set.seed(1)
DATA <- file.path(Sys.getenv("HOME"), "clozapine", "data")
OUT  <- file.path(Sys.getenv("HOME"), "clozapine", "out")
dir.create(DATA, recursive=TRUE, showWarnings=FALSE)
dir.create(OUT,  recursive=TRUE, showWarnings=FALSE)
options(timeout = 3600)

stopifnot(all(sapply(c("DESeq2","limma","fgsea","msigdbr"), requireNamespace, quietly=TRUE)))
cat("environment OK\n")

environment OK


## 0b. GEO retrieval helpers

`geo_suppl()` replaces `getGEOSuppFiles()`; `geo_series_matrix()` replaces
`getGEO(GSEMatrix=TRUE)`. Both are base R over HTTPS.

In [2]:
GEO_FTP <- "https://ftp.ncbi.nlm.nih.gov/geo/series"

geo_stub <- function(acc) {
  n <- sub("^GSE", "", acc)
  paste0("GSE", substr(n, 1, nchar(n) - 3), "nnn")
}

geo_listdir <- function(url) {
  txt   <- paste(readLines(url, warn = FALSE), collapse = "\n")
  hits  <- regmatches(txt, gregexpr('href="[^"]+"', txt))[[1]]
  files <- sub('^href="', "", sub('"$', "", hits))
  unique(files[!grepl("^/|^\\?|\\.\\.", files)])
}

geo_suppl <- function(acc, dest = DATA, pattern = NULL) {
  dir  <- file.path(dest, acc); dir.create(dir, recursive = TRUE, showWarnings = FALSE)
  base <- sprintf("%s/%s/%s/suppl/", GEO_FTP, geo_stub(acc), acc)
  files <- geo_listdir(base)
  if (!is.null(pattern)) files <- grep(pattern, files, value = TRUE)
  message("Supplementary files at ", acc, ":\n  ", paste(files, collapse = "\n  "))
  for (f in files) {
    out <- file.path(dir, f)
    if (file.exists(out) && file.size(out) > 0) { message("cached: ", f); next }
    message("downloading: ", f)
    download.file(paste0(base, f), out, mode = "wb", quiet = TRUE)
  }
  invisible(list.files(dir, full.names = TRUE))
}

geo_series_matrix <- function(acc, dest = DATA) {
  dir <- file.path(dest, acc); dir.create(dir, recursive = TRUE, showWarnings = FALSE)
  fn  <- sprintf("%s_series_matrix.txt.gz", acc)
  out <- file.path(dir, fn)
  if (!file.exists(out) || file.size(out) == 0) {
    url <- sprintf("%s/%s/%s/matrix/%s", GEO_FTP, geo_stub(acc), acc, fn)
    message("downloading series matrix for ", acc)
    download.file(url, out, mode = "wb", quiet = TRUE)
  }
  ln <- readLines(gzfile(out), warn = FALSE)

  smp  <- grep("^!Sample_", ln, value = TRUE)
  rows <- lapply(smp, function(s) {
    parts <- strsplit(s, "\t")[[1]]
    list(key = sub("^!", "", parts[1]), vals = gsub('^"|"$', "", parts[-1]))
  })
  n <- max(vapply(rows, function(r) length(r$vals), integer(1)))
  pheno <- data.frame(row.names = seq_len(n))
  for (r in rows) {
    v <- r$vals; length(v) <- n
    k <- r$key; i <- 1
    while (k %in% names(pheno)) { i <- i + 1; k <- paste0(r$key, ".", i) }
    pheno[[k]] <- v
  }

  b <- grep("^!series_matrix_table_begin", ln); e <- grep("^!series_matrix_table_end", ln)
  expr <- NULL
  if (length(b) && length(e) && e > b + 1) {
    tab  <- read.delim(text = paste(ln[(b+1):(e-1)], collapse = "\n"),
                       header = TRUE, check.names = FALSE, row.names = 1)
    expr <- as.matrix(tab)
  }
  list(pheno = pheno, expr = expr)
}

cat("helpers loaded\n")

helpers loaded


## 1. GSE262419 — retrieve

In [3]:
f <- geo_suppl("GSE262419")
print(f)

Supplementary files at GSE262419:
  GSE262419_Plate01.csv.gz
  GSE262419_Plate02.csv.gz
  GSE262419_Plate03.csv.gz
  GSE262419_Plate04.csv.gz
  GSE262419_Plate05.csv.gz
  GSE262419_Plate06.csv.gz
  GSE262419_Plate07.csv.gz
  GSE262419_Plate08.csv.gz
  GSE262419_Plate09.csv.gz
  GSE262419_Plate10.csv.gz
  GSE262419_Plate11.csv.gz
  GSE262419_Plate12.csv.gz
  GSE262419_Plate13.csv.gz
  GSE262419_Plate14.csv.gz
  GSE262419_Plate15.csv.gz
  GSE262419_Plate16.csv.gz
  GSE262419_hash.csv.gz
  https://www.hhs.gov/vulnerability-disclosure-policy/index.html

downloading: GSE262419_Plate01.csv.gz

downloading: GSE262419_Plate02.csv.gz

downloading: GSE262419_Plate03.csv.gz

downloading: GSE262419_Plate04.csv.gz

downloading: GSE262419_Plate05.csv.gz

downloading: GSE262419_Plate06.csv.gz

downloading: GSE262419_Plate07.csv.gz

downloading: GSE262419_Plate08.csv.gz

downloading: GSE262419_Plate09.csv.gz

downloading: GSE262419_Plate10.csv.gz

downloading: GSE262419_Plate11.csv.gz

downloading: GS

 [1] "/home/mnho/clozapine/data/GSE262419/GSE262419_hash.csv.gz"   
 [2] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate01.csv.gz"
 [3] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate02.csv.gz"
 [4] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate03.csv.gz"
 [5] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate04.csv.gz"
 [6] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate05.csv.gz"
 [7] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate06.csv.gz"
 [8] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate07.csv.gz"
 [9] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate08.csv.gz"
[10] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate09.csv.gz"
[11] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate10.csv.gz"
[12] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate11.csv.gz"
[13] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate12.csv.gz"
[14] "/home/mnho/clozapine/data/GSE262419/GSE262419_Plate13.csv.gz"
[15] "/home/mnho/clozapine/data/GSE262419/GSE262

**INSPECT.** Confirm the count-matrix layout before proceeding.

In [4]:
cnt_file <- grep("Plate1[56]|counts", f, value=TRUE)[1]
cat("using:", cnt_file, "\n\n")
raw <- read.csv(gzfile(cnt_file), row.names=1, check.names=FALSE)
cat("dim:", dim(raw), "\n"); print(raw[1:5, 1:6])
cat("\ncolumn names (first 20):\n"); print(head(colnames(raw), 20))

using: /home/mnho/clozapine/data/GSE262419/GSE262419_Plate15.csv.gz 

dim: 22537 286 
            Plate15-B02 Plate15-B03 Plate15-B04 Plate15-B05 Plate15-B06
A1BG_25586           18           0           0           0          17
A1CF_87905           43           0           0           0           0
A2ML1_18347           0           0           0           0           0
A2M_1                11           0          24          16          17
A2M_12371             0           0          15          12          41
            Plate15-B07
A1BG_25586           27
A1CF_87905            0
A2ML1_18347           0
A2M_1                39
A2M_12371            19

column names (first 20):
 [1] "Plate15-B02" "Plate15-B03" "Plate15-B04" "Plate15-B05" "Plate15-B06"
 [6] "Plate15-B07" "Plate15-B08" "Plate15-B09" "Plate15-B10" "Plate15-B11"
[11] "Plate15-B12" "Plate15-B13" "Plate15-B14" "Plate15-B15" "Plate15-B16"
[16] "Plate15-B17" "Plate15-B18" "Plate15-B19" "Plate15-B20" "Plate15-B21"


**INSPECT.** Sample metadata. Paste this output back before building `meta`.

In [5]:
sm <- geo_series_matrix("GSE262419")
cat("samples in series matrix:", nrow(sm$pheno), "\n\n")
print(names(sm$pheno))
show <- grep("title|characteristics|source|description", names(sm$pheno),
             ignore.case=TRUE, value=TRUE)
print(head(sm$pheno[, show, drop=FALSE], 10))

downloading series matrix for GSE262419



samples in series matrix: 4558 

 [1] "Sample_title"                   "Sample_geo_accession"          
 [3] "Sample_status"                  "Sample_submission_date"        
 [5] "Sample_last_update_date"        "Sample_type"                   
 [7] "Sample_channel_count"           "Sample_source_name_ch1"        
 [9] "Sample_organism_ch1"            "Sample_characteristics_ch1"    
[11] "Sample_characteristics_ch1.2"   "Sample_molecule_ch1"           
[13] "Sample_extract_protocol_ch1"    "Sample_extract_protocol_ch1.2" 
[15] "Sample_taxid_ch1"               "Sample_description"            
[17] "Sample_data_processing"         "Sample_data_processing.2"      
[19] "Sample_data_processing.3"       "Sample_data_processing.4"      
[21] "Sample_data_processing.5"       "Sample_data_processing.6"      
[23] "Sample_platform_id"             "Sample_contact_name"           
[25] "Sample_contact_department"      "Sample_contact_institute"      
[27] "Sample_contact_address"         "Sampl

In [6]:
# Fill in from the inspection above. One row per column of `raw`.
meta <- data.frame(
  sample = colnames(raw),
  plate  = NA_character_,   # "15" or "16"
  drug   = NA_character_,   # "clozapine","haloperidol","risperidone","DMSO"
  conc   = NA_real_,        # 0.2, 1, 10; NA for vehicle
  stringsAsFactors = FALSE
)

stopifnot(!any(is.na(meta$drug)), !any(is.na(meta$plate)))
with(meta, table(plate, drug))
# Expect haloperidol on plate 15; clozapine + risperidone on plate 16; DMSO on both.

ERROR: Error: !any(is.na(meta$drug)) is not TRUE


## 2. Differential expression

Adjusted p < 0.05 **and** mean normalized expression >= 100, against plate-matched DMSO.

In [ ]:
run_contrast <- function(raw, meta, the_drug, the_conc, the_plate) {
  keep <- meta$plate == the_plate &
          ((meta$drug == the_drug & meta$conc == the_conc) | meta$drug == "DMSO")
  m <- meta[keep, ]
  m$group <- factor(ifelse(m$drug == "DMSO", "vehicle", "treated"),
                    levels=c("vehicle","treated"))
  cts <- as.matrix(raw[, m$sample, drop=FALSE]); mode(cts) <- "integer"
  dds <- DESeq(DESeqDataSetFromMatrix(cts, colData=m, design = ~ group), quiet=TRUE)
  res <- as.data.frame(results(dds, contrast=c("group","treated","vehicle")))
  res$gene <- rownames(res)
  res
}

filter_degs <- function(res, padj_cut = 0.05, basemean_cut = 100)
  subset(res, !is.na(padj) & padj < padj_cut & baseMean >= basemean_cut)

In [ ]:
res_clz10 <- run_contrast(raw, meta, "clozapine",   10,  "16")
res_clz1  <- run_contrast(raw, meta, "clozapine",    1,  "16")
res_clz02 <- run_contrast(raw, meta, "clozapine",  0.2,  "16")
res_hal10 <- run_contrast(raw, meta, "haloperidol", 10,  "15")
res_ris10 <- run_contrast(raw, meta, "risperidone", 10,  "16")

deg <- list(clozapine_10   = filter_degs(res_clz10),
            clozapine_1    = filter_degs(res_clz1),
            clozapine_0.2  = filter_degs(res_clz02),
            haloperidol_10 = filter_degs(res_hal10),
            risperidone_10 = filter_degs(res_ris10))
sapply(deg, nrow)

# --- MANUSCRIPT TARGETS: haloperidol 95, clozapine 9, risperidone 0,
#     clozapine at 0.2 and 1 uM both 0.
# A mismatch means the manuscript numbers do not reproduce. Stop and
# investigate. Do not tune thresholds to fit.

In [ ]:
target9 <- c(HIST1H1B=1.66, HIST2H4B=1.24, HIST1H4I=1.25, NUSAP1=1.13,
             TNFRSF12A=-2.14, EGLN3=-2.12, VEGFA=-1.75, SH3D21=-1.44, PRSS45P=-1.39)
got <- deg$clozapine_10
cmp <- data.frame(gene=names(target9), reported=as.numeric(target9))
cmp$recomputed <- got$log2FoldChange[match(cmp$gene, got$gene)]
cmp$delta <- round(cmp$recomputed - cmp$reported, 3)
cmp

## 3. Cardiac gene-set enrichment

Universe restricted to genes meeting the same expression threshold within each drug's own
comparison, which is why eligible set sizes differ between plates.

In [ ]:
sets <- msigdbr(species="Homo sapiens", collection="C5", subcollection="GO:BP")
pick <- function(nm) unique(sets$gene_symbol[sets$gs_name == nm])
GS <- list(
  cardiac_muscle_contraction        = pick("GOBP_CARDIAC_MUSCLE_CONTRACTION"),
  sarcomere_organization            = pick("GOBP_SARCOMERE_ORGANIZATION"),
  cardiac_muscle_tissue_development = pick("GOBP_CARDIAC_MUSCLE_TISSUE_DEVELOPMENT"))
sapply(GS, length)

In [ ]:
ora <- function(res_full, deg_tbl, gene_set, basemean_cut = 100) {
  universe <- res_full$gene[!is.na(res_full$padj) & res_full$baseMean >= basemean_cut]
  set_in_u <- intersect(gene_set, universe)
  hits     <- intersect(deg_tbl$gene, set_in_u)
  N <- length(universe); K <- length(set_in_u); n <- nrow(deg_tbl); k <- length(hits)
  data.frame(universe=N, set_eligible=K, degs=n, observed=k,
             expected=round(n*K/N, 2),
             p = if (n==0 || K==0) NA_real_ else phyper(k-1, K, N-K, n, lower.tail=FALSE),
             genes = paste(sort(hits), collapse=", "))
}

enrich <- bind_rows(
  cbind(drug="haloperidol", set="cardiac_muscle_contraction",
        ora(res_hal10, deg$haloperidol_10, GS$cardiac_muscle_contraction)),
  cbind(drug="haloperidol", set="sarcomere_organization",
        ora(res_hal10, deg$haloperidol_10, GS$sarcomere_organization)),
  cbind(drug="clozapine",   set="cardiac_muscle_contraction",
        ora(res_clz10, deg$clozapine_10,   GS$cardiac_muscle_contraction)),
  cbind(drug="clozapine",   set="sarcomere_organization",
        ora(res_clz10, deg$clozapine_10,   GS$sarcomere_organization)))
enrich

# --- MANUSCRIPT TARGETS (Table 3) ---
# haloperidol/contraction: eligible 37, observed 8, expected 2.20, p 0.0011
#   CSRP3, MYH6, MYL3, MYL4, PDE4B, PRKACA, SNTA1, TNNC1
# haloperidol/sarcomere:   eligible 20, observed 4, expected 1.19, p 0.027
#   CSRP3, LMOD2, MYH6, OBSCN
# clozapine/contraction:   eligible 32, observed 0, expected 0.30
# clozapine/sarcomere:     eligible 18, observed 0, expected 0.17

## 4. Plate sensitivity analysis

New; not in the current draft. Haloperidol is on plate 15 and clozapine on plate 16, so
the headline cross-drug comparison crosses a batch boundary. Check B first: it is the
cleaner diagnostic. If the vehicle-only plate effect is large relative to the drug effects,
the cross-plate comparison cannot carry the primary claim regardless of what A shows.

In [ ]:
mv <- meta[meta$drug == "DMSO", ]; mv$plate <- factor(mv$plate)
ddsv <- DESeq(DESeqDataSetFromMatrix(as.matrix(raw[, mv$sample]),
                                     colData=mv, design = ~ plate), quiet=TRUE)
resv <- as.data.frame(results(ddsv, contrast=c("plate","15","16"))); resv$gene <- rownames(resv)

plate_degs <- filter_degs(resv)
cat("Vehicle-only genes differing between plate 15 and 16:", nrow(plate_degs), "\n")
cat("Overlap with haloperidol DEGs:",
    length(intersect(plate_degs$gene, deg$haloperidol_10$gene)),
    "of", nrow(deg$haloperidol_10), "\n")

contraction_hits <- intersect(deg$haloperidol_10$gene, GS$cardiac_muscle_contraction)
cat("Contraction-set hits also plate-confounded:",
    paste(intersect(plate_degs$gene, contraction_hits), collapse=", "), "\n")

In [ ]:
md2 <- meta[(meta$drug=="haloperidol" & meta$conc==10) |
            (meta$drug=="clozapine"   & meta$conc==10) | meta$drug=="DMSO", ]
md2$drug  <- factor(md2$drug, levels=c("DMSO","clozapine","haloperidol"))
md2$plate <- factor(md2$plate)
print(with(md2, table(plate, drug)))
cat("\nIf a drug appears on only one plate, the plate term is aliased with it.\n",
    "Report that as a limitation rather than presenting an adjusted estimate as clean.\n")

## 5. GSE244740 — human validation

In [7]:
geo_suppl("GSE244740")
sm244 <- geo_series_matrix("GSE244740")
cat("samples:", nrow(sm244$pheno), "\n"); print(names(sm244$pheno))
# INSPECT, then build meta244 and run the identical DESeq2 workflow.
# TARGETS: TNFRSF12A -1.66 (padj 7.9e-74, baseMean 421); EGLN3 -1.70 (padj 4.0e-29);
#          VEGFA -0.46 (padj 0.17, baseMean 26) - does not replicate.

Supplementary files at GSE244740:
  GSE244740_processed_data_counts.txt.gz
  https://www.hhs.gov/vulnerability-disclosure-policy/index.html

downloading: GSE244740_processed_data_counts.txt.gz

downloading: https://www.hhs.gov/vulnerability-disclosure-policy/index.html

Warning message in download.file(paste0(base, f), out, mode = "wb", quiet = TRUE):
“URL https://ftp.ncbi.nlm.nih.gov/geo/series/GSE244nnn/GSE244740/suppl/https://www.hhs.gov/vulnerability-disclosure-policy/index.html: cannot open destfile '/home/mnho/clozapine/data/GSE244740/https://www.hhs.gov/vulnerability-disclosure-policy/index.html', reason 'No such file or directory'”
Warning message in download.file(paste0(base, f), out, mode = "wb", quiet = TRUE):
“download had nonzero exit status”
downloading series matrix for GSE244740



samples: 3072 
 [1] "Sample_title"                   "Sample_geo_accession"          
 [3] "Sample_status"                  "Sample_submission_date"        
 [5] "Sample_last_update_date"        "Sample_type"                   
 [7] "Sample_channel_count"           "Sample_source_name_ch1"        
 [9] "Sample_organism_ch1"            "Sample_characteristics_ch1"    
[11] "Sample_characteristics_ch1.2"   "Sample_characteristics_ch1.3"  
[13] "Sample_characteristics_ch1.4"   "Sample_treatment_protocol_ch1" 
[15] "Sample_growth_protocol_ch1"     "Sample_molecule_ch1"           
[17] "Sample_extract_protocol_ch1"    "Sample_extract_protocol_ch1.2" 
[19] "Sample_taxid_ch1"               "Sample_description"            
[21] "Sample_data_processing"         "Sample_data_processing.2"      
[23] "Sample_data_processing.3"       "Sample_data_processing.4"      
[25] "Sample_data_processing.5"       "Sample_data_processing.6"      
[27] "Sample_platform_id"             "Sample_contact_name"   

## 6. GSE59905 — rat heart, DrugMatrix

Oral clozapine 95 mg/kg, nine animals, days 1/3/5; 32 zero-dose vehicle-matched control
hearts. Platform GPL5425, limma. The series matrix carries the expression table, so no
supplementary download is needed.

In [8]:
sm59 <- geo_series_matrix("GSE59905")
cat("samples:", nrow(sm59$pheno), "\n")
cat("expression:", if (is.null(sm59$expr)) "absent" else paste(dim(sm59$expr), collapse=" x "), "\n\n")
print(names(sm59$pheno))
print(head(sm59$pheno[, grep("title|characteristics", names(sm59$pheno),
                             ignore.case=TRUE, value=TRUE), drop=FALSE], 12))

downloading series matrix for GSE59905

Warning message in download.file(url, out, mode = "wb", quiet = TRUE):
“cannot open URL 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE59nnn/GSE59905/matrix/GSE59905_series_matrix.txt.gz': HTTP status was '404 Not Found'”


ERROR: Error in download.file(url, out, mode = "wb", quiet = TRUE): cannot open URL 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE59nnn/GSE59905/matrix/GSE59905_series_matrix.txt.gz'


In [ ]:
# After identifying groups from the inspection above:
# grp    <- factor(ifelse(<clozapine samples>, "clz", "ctl"), levels=c("ctl","clz"))
# fit    <- eBayes(lmFit(sm59$expr, model.matrix(~ grp)))
# tt     <- topTable(fit, coef=2, number=Inf)
#
# Probe -> symbol via the platform annotation (also XML-free):
# download.file("https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL5nnn/GPL5425/annot/GPL5425.annot.gz",
#               file.path(DATA,"GPL5425.annot.gz"), mode="wb")
#
# TARGETS: TNFRSF12A -1.48 (padj 3.5e-4); EGLN3 -0.64 (padj 6.6e-3); VEGFA +0.08 n.s.

## 7. Leave-one-out robustness (TNFRSF12A)

In [ ]:
clz_wells <- meta$sample[meta$drug=="clozapine" & meta$conc==10 & meta$plate=="16"]
loo <- lapply(clz_wells, function(drop) {
  r <- run_contrast(raw, meta[meta$sample != drop, ], "clozapine", 10, "16")
  r[r$gene == "TNFRSF12A", c("log2FoldChange","padj")]
})
names(loo) <- paste("dropped", clz_wells)
do.call(rbind, loo)
# Draft reports padj ranging 4.3e-4 to 0.124 across leave-one-out.

## 8. Figure 1

In [ ]:
conc_tbl <- bind_rows(
  cbind(conc=0.2, res_clz02[res_clz02$gene %in% names(target9),
                            c("gene","log2FoldChange","padj","baseMean")]),
  cbind(conc=1,   res_clz1 [res_clz1$gene  %in% names(target9),
                            c("gene","log2FoldChange","padj","baseMean")]),
  cbind(conc=10,  res_clz10[res_clz10$gene %in% names(target9),
                            c("gene","log2FoldChange","padj","baseMean")])) %>%
  mutate(sig = !is.na(padj) & padj < 0.05 & baseMean >= 100,
         dir = ifelse(gene %in% names(target9)[target9 > 0], "up", "down"),
         gene = factor(gene, levels=names(target9)))

p1 <- ggplot(conc_tbl, aes(conc, log2FoldChange, colour=dir)) +
  annotate("rect", xmin=1.05, xmax=1.80, ymin=-Inf, ymax=Inf, fill="grey85", alpha=.6) +
  geom_hline(yintercept=0, colour="grey40") +
  geom_line(linewidth=.6) +
  geom_point(aes(shape=sig), size=2.4, fill="white", stroke=.9) +
  scale_shape_manual(values=c(`FALSE`=21, `TRUE`=19),
                     labels=c("Not significant","Adjusted p < 0.05")) +
  scale_colour_manual(values=c(down="#B2182B", up="#2166AC"), guide="none") +
  scale_x_log10(breaks=c(0.2,1,10), labels=c("0.2","1","10")) +
  facet_wrap(~gene, ncol=3) +
  labs(x="Clozapine concentration (\u00b5M, log scale)",
       y=expression(log[2]~fold-change~vs~vehicle), shape=NULL) +
  theme_bw(base_size=11) +
  theme(legend.position="bottom", strip.text=element_text(face="bold.italic"))

ggsave(file.path(OUT,"fig1_concentration.png"), p1, width=9, height=7.5, dpi=600)
p1

## 9. Supplementary tables and provenance

In [ ]:
si <- sessionInfo()
pkgs <- c("DESeq2","limma","fgsea","msigdbr","ggplot2")
s3 <- data.frame(
  Component = c("R","Bioconductor",pkgs,"Platform","Operating system"),
  Version = c(paste(R.version$major, R.version$minor, sep="."),
              as.character(BiocManager::version()),
              vapply(pkgs, function(p) as.character(packageVersion(p)), character(1)),
              R.version$platform, si$running))
write.csv(s3, file.path(OUT,"TableS3_software.csv"), row.names=FALSE)
print(s3, row.names=FALSE)
cat("\nThis is what Methods 2.8 must say. GEOquery is NOT used; data were retrieved\n",
    "directly from the NCBI GEO FTP archive using base R.\n")

In [ ]:
elig <- function(res_full, gene_set, basemean_cut=100) {
  u <- res_full$gene[!is.na(res_full$padj) & res_full$baseMean >= basemean_cut]
  length(intersect(gene_set, u))
}
s1a <- do.call(rbind, lapply(names(GS), function(gs) data.frame(
  gene_set=gs, total_members=length(GS[[gs]]),
  eligible_haloperidol=elig(res_hal10, GS[[gs]]),
  eligible_clozapine  =elig(res_clz10, GS[[gs]]))))
write.csv(s1a, file.path(OUT,"TableS1a_geneset_summary.csv"), row.names=FALSE)
print(s1a, row.names=FALSE)
# TARGETS: contraction 37 vs 32; sarcomere 20 vs 18.

u_hal <- res_hal10$gene[!is.na(res_hal10$padj) & res_hal10$baseMean >= 100]
u_clz <- res_clz10$gene[!is.na(res_clz10$padj) & res_clz10$baseMean >= 100]
s1b <- do.call(rbind, lapply(names(GS), function(gs) {
  g <- sort(GS[[gs]])
  data.frame(gene_set=gs, gene_symbol=g,
             eligible_haloperidol = g %in% u_hal,
             eligible_clozapine   = g %in% u_clz,
             deg_haloperidol_10uM = g %in% deg$haloperidol_10$gene,
             deg_clozapine_10uM   = g %in% deg$clozapine_10$gene)
}))
write.csv(s1b, file.path(OUT,"TableS1b_geneset_membership.csv"), row.names=FALSE)
cat("S1b rows:", nrow(s1b), "\n")

writeLines(capture.output(sessionInfo()), file.path(OUT,"sessionInfo.txt"))
saveRDS(list(deg=deg, enrich=enrich, meta=meta), file.path(OUT,"results.rds"))
cat("Run completed:", format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z"), "\n")